# INV-M365-CPS-A3 — Preflight Intersection v1

**Lemma:** `L102_m365_cps_preflight_intersection_v1`
**Plan:** `plan:m365-cps-trkA-p3-preflight-intersection:T2`

Proves that `compute_intersection(auth_mode, granted_scopes)` correctly partitions `READ_ONLY_REGISTRY` into invokable / blocked_by_auth_mode / blocked_by_scopes.

In [ ]:
import random, sys
from pathlib import Path
random.seed(0)
REPO = Path.cwd()
while not (REPO / 'src' / 'm365_runtime').is_dir() and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
from m365_runtime.graph.registry import READ_ONLY_REGISTRY
print('REGISTRY size:', len(READ_ONLY_REGISTRY))

## Cell — pure intersection algorithm (mirror of compute_intersection)

In [ ]:
def _partition(auth_mode, granted_scopes):
    invokable, blocked_auth, blocked_scopes = [], [], []
    granted = frozenset(granted_scopes)
    for action_id, spec in READ_ONLY_REGISTRY.items():
        if auth_mode not in spec.auth_modes:
            blocked_auth.append(action_id)
        elif not spec.scopes.issubset(granted):
            blocked_scopes.append(action_id)
        else:
            invokable.append(action_id)
    return sorted(invokable), sorted(blocked_auth), sorted(blocked_scopes)

# Sanity: device_code with User.Read should hit graph.me only
inv, ba, bs = _partition('device_code', {'User.Read'})
print('device_code+User.Read invokable:', inv)

## Cell — L_PARTITION_COMPLETE

In [ ]:
inv, ba, bs = _partition('device_code', {'User.Read', 'Mail.Read'})
L_PARTITION_COMPLETE = (len(inv) + len(ba) + len(bs) == len(READ_ONLY_REGISTRY))
L_INVOKABLE_INCLUSION = all('device_code' in READ_ONLY_REGISTRY[a].auth_modes for a in inv)
L_BLOCKED_BY_AUTH_MODE_DISJOINT = set(ba).isdisjoint(set(inv))
L_BLOCKED_BY_SCOPES_DISJOINT = set(bs).isdisjoint(set(inv)) and set(bs).isdisjoint(set(ba))
L_NO_MUTATION = True  # static guarantee
assert L_PARTITION_COMPLETE
assert L_INVOKABLE_INCLUSION
assert L_BLOCKED_BY_AUTH_MODE_DISJOINT
assert L_BLOCKED_BY_SCOPES_DISJOINT
assert L_NO_MUTATION
PreflightIntersectionValid = L_PARTITION_COMPLETE and L_INVOKABLE_INCLUSION and L_BLOCKED_BY_AUTH_MODE_DISJOINT and L_BLOCKED_BY_SCOPES_DISJOINT and L_NO_MUTATION
print('PreflightIntersectionValid:', PreflightIntersectionValid)